# Huấn Luyện Benchmark Mô Hình UC-TTT & TTT Baseline Trên Google Colab
## (User-Credibility Temporal Tree Transformer vs Wu et al. PLOS ONE 2025)

Notebook này được thiết kế để chạy thực nghiệm phát hiện tin đồn trên mạng xã hội PHEME trên GPU đám mây của **Google Colab** (Tesla T4 / A100 / V100).

### 🌟 Các cải tiến chuẩn bài báo được tích hợp:
1. **Text Embedding**: `bert-base-uncased` (768 chiều, Max-Pooling theo công thức (1) của bài báo).
2. **Cửa sổ thời gian**: Chia động theo số bài viết ($k=8$ posts/cửa sổ), BiGRU hỗ trợ packed sequences cho độ dài biến thiên.
3. **Kiểm định LOEO**: 5 sự kiện đối chứng chuẩn Wu et al. 2025 (*charliehebdo, ferguson, germanwings-crash, ottawashooting, sydney-siege*).

---

### Bước 1: Kiểm tra phần cứng GPU được cấp phát

In [ ]:
!nvidia-smi

### Bước 2: Kết nối Google Drive để đồng bộ dữ liệu và lưu checkpoint

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Bước 3: Clone Codebase từ GitHub (Nhánh `reimplement-TTT-TTT_UC`)

In [ ]:
%cd /content
!rm -rf repo
!git clone -b reimplement-TTT-TTT_UC https://github.com/tphu2029/SNA_EnhancingTTT-via-User-Credibility-Integration-for-Social-Media-Rumor-Detection.git repo
%cd /content/repo
!git status

### Bước 4: Cài đặt các thư viện cần thiết

In [ ]:
!pip install -q "transformers>=4.30.0,<4.49.0" scikit-learn pandas networkx tqdm

### Bước 5: Nạp dữ liệu từ Google Drive

Hệ thống sẽ tự động tìm kiếm thư mục dữ liệu PHEME (`pheme_processed.pkl` và `text_embeddings_bert.pt`) trên Google Drive của bạn (hỗ trợ `MXH/PHEME_DATA`, `PHEME_DATA`, hoặc tự động quét toàn bộ Drive).

> ⚠️ **LƯU Ý NẾU THƯ MỤC NẰM Ở 'ĐƯỢC CHIA SẺ VỚI TÔI' (Shared with me)**:
> Google Colab chỉ đọc được thư mục nằm trong **'Drive của tôi' (My Drive)**. Nếu thư mục `PHEME_DATA` do người khác chia sẻ, bạn hãy:
> 1. Mở Google Drive trên trình duyệt (https://drive.google.com) -> Vào **'Được chia sẻ với tôi'**
> 2. Click chuột phải vào thư mục `PHEME_DATA` (hoặc `MXH`) -> Chọn **Tổ chức (Organize)** -> Chọn **Thêm lối tắt (Add shortcut)**
> 3. Chọn đích đến là **'Drive của tôi' (My Drive)** -> Bấm **Thêm**

In [ ]:
import os
import shutil

print("🔍 Đang tìm kiếm dữ liệu PHEME trên Google Drive...")

# Danh sách đường dẫn dự kiến phổ biến
CANDIDATE_DIRS = [
    '/content/drive/MyDrive/MXH/PHEME_DATA',
    '/content/drive/MyDrive/PHEME_DATA',
    '/content/drive/MyDrive/Colab Notebooks/PHEME_DATA',
    '/content/drive/MyDrive/PHEME',
    '/content/drive/MyDrive/MXH',
]

DRIVE_DATA_DIR = None

# 1. Thử các đường dẫn mặc định
for d in CANDIDATE_DIRS:
    if os.path.exists(d) and (os.path.exists(os.path.join(d, 'pheme_processed.pkl')) or os.path.exists(os.path.join(d, 'text_embeddings_bert.pt'))):
        DRIVE_DATA_DIR = d
        print(f"-> Đã tìm thấy dữ liệu tại: {DRIVE_DATA_DIR}")
        break

# 2. Nếu không thấy, quét đệ quy các thư mục trong /content/drive/MyDrive
if DRIVE_DATA_DIR is None and os.path.exists('/content/drive/MyDrive'):
    print("-> Không thấy ở đường dẫn chuẩn, đang quét tự động /content/drive/MyDrive để tìm file...")
    for root, dirs, files in os.walk('/content/drive/MyDrive'):
        if 'pheme_processed.pkl' in files or 'text_embeddings_bert.pt' in files:
            DRIVE_DATA_DIR = root
            print(f"-> [TÌM THẤY TỰ ĐỘNG] Dữ liệu nằm tại: {DRIVE_DATA_DIR}")
            break

# 3. Quét rộng hơn toàn bộ /content/drive nếu vẫn chưa thấy
if DRIVE_DATA_DIR is None and os.path.exists('/content/drive'):
    for root, dirs, files in os.walk('/content/drive'):
        if 'pheme_processed.pkl' in files or 'text_embeddings_bert.pt' in files:
            DRIVE_DATA_DIR = root
            print(f"-> [TÌM THẤY TỰ ĐỘNG] Dữ liệu nằm tại: {DRIVE_DATA_DIR}")
            break

os.makedirs('data/processed', exist_ok=True)

if DRIVE_DATA_DIR is None:
    msg = (
        "\n" + "="*75 + "\n"
        "❌ LỖI: KHÔNG TÌM THẤY 'pheme_processed.pkl' TRÊN GOOGLE DRIVE!\n"
        "="*75 + "\n"
        "📌 NGUYÊN NHÂN:\n"
        "Thư mục 'PHEME_DATA' (hoặc 'MXH') đang nằm ở mục 'Được chia sẻ với tôi' (Shared with me).\n"
        "Google Colab KHÔNG THỂ truy cập thư mục được chia sẻ nếu chưa tạo Lối tắt vào 'Drive của tôi'.\n\n"
        "👉 HƯỚNG DẪN KHẮC PHỤC (Chỉ 10 giây):\n"
        "1. Mở Google Drive trên trình duyệt (https://drive.google.com)\n"
        "2. Vào mục 'Được chia sẻ với tôi' (Shared with me) ở danh mục bên trái\n"
        "3. Click chuột PHẢI vào thư mục 'PHEME_DATA' (hoặc 'MXH')\n"
        "4. Chọn 'Tổ chức' (Organize) -> Chọn 'Thêm lối tắt' (Add shortcut)\n"
        "5. Chọn 'Drive của tôi' (My Drive) -> Bấm nút 'Thêm' (Add)\n"
        "6. Quay lại Colab và BẤM CHẠY LẠI CELL NÀY!\n"
        "="*75 + "\n"
    )
    print(msg)
    raise FileNotFoundError("Chưa tìm thấy dữ liệu trên Google Drive! Hãy tạo 'Lối tắt' (Shortcut) vào 'Drive của tôi' theo hướng dẫn ở trên.")

print(f"\n📂 Thư mục nguồn Google Drive: {DRIVE_DATA_DIR}")

# 1. Copy pheme_processed.pkl
pkl_src = os.path.join(DRIVE_DATA_DIR, 'pheme_processed.pkl')
pkl_dst = 'data/processed/pheme_processed.pkl'
if os.path.exists(pkl_src):
    print("Đang nạp pheme_processed.pkl...")
    shutil.copyfile(pkl_src, pkl_dst)
    size_mb = os.path.getsize(pkl_dst) / 1024 / 1024
    print(f"✅ Đã nạp thành công: {pkl_dst} ({size_mb:.1f} MB)")
else:
    print(f"⚠️ Cảnh báo: Không tìm thấy pheme_processed.pkl trong {DRIVE_DATA_DIR}")

# 2. Copy text_embeddings_bert.pt
bert_src = os.path.join(DRIVE_DATA_DIR, 'text_embeddings_bert.pt')
bert_dst = 'data/processed/text_embeddings_bert.pt'
if os.path.exists(bert_src):
    print("Đang nạp text_embeddings_bert.pt...")
    shutil.copyfile(bert_src, bert_dst)
    size_mb = os.path.getsize(bert_dst) / 1024 / 1024
    print(f"✅ Đã nạp thành công: {bert_dst} ({size_mb:.1f} MB)")
    print("\n🎉 THÀNH CÔNG! Đã nạp đầy đủ dữ liệu cây PHEME và text embeddings!")
    print("⏩ Bạn CÓ THỂ BỎ QUA Bước 5.2 và nhảy thẳng xuống Bước 6 để huấn luyện!")
else:
    print("\nℹ️ [Thông báo] Chưa có sẵn text_embeddings_bert.pt trên Drive.")
    print("👉 Hãy chạy Bước 5.2 bên dưới để trích xuất embedding từ mô hình BERT.")

### Bước 5.2: (TÙY CHỌN - BỎ QUA NẾU ĐÃ NẠP XONG Ở BƯỚC 5) Trích xuất BERT Embedding
> 💡 **Lưu ý quan trọng**: Nếu ở **Bước 5** bạn đã nạp thành công file `text_embeddings_bert.pt` từ Google Drive (thấy thông báo màu xanh ✅), **BẠN KHÔNG CẦN CHẠY BƯỚC NÀY**! Hãy nhảy thẳng xuống **Bước 6** để huấn luyện.

In [ ]:
if os.path.exists("data/processed/text_embeddings_bert.pt"):
    print("✅ File 'data/processed/text_embeddings_bert.pt' ĐÃ CÓ SẴN (đã nạp từ Google Drive)!")
    print("⏩ Không cần trích xuất lại. Bạn hãy chuyển thẳng sang Bước 6 để huấn luyện mô hình!")
else:
    if not os.path.exists("data/processed/pheme_processed.pkl"):
        raise FileNotFoundError(
            "❌ Chưa có 'data/processed/pheme_processed.pkl'!\n"
            "Vui lòng chạy lại Bước 5 để nạp file dữ liệu từ Google Drive trước khi trích xuất embedding."
        )
    print("⏳ Không tìm thấy embedding có sẵn. Bắt đầu trích xuất 768-d BERT embeddings với Max-Pooling trên GPU (~2 phút)...\n")
    !python experiments/extract_text_embeddings.py \
        --model_name bert-base-uncased \
        --batch_size 128 \
        --pooling max \
        --output data/processed/text_embeddings_bert.pt
    
    if os.path.exists("data/processed/text_embeddings_bert.pt"):
        print("\n✅ Đã trích xuất thành công text_embeddings_bert.pt!")
        if DRIVE_DATA_DIR and os.path.exists(DRIVE_DATA_DIR):
            print("💾 Đang sao lưu text_embeddings_bert.pt lên Google Drive...")
            try:
                shutil.copyfile("data/processed/text_embeddings_bert.pt", os.path.join(DRIVE_DATA_DIR, "text_embeddings_bert.pt"))
                print("✅ Đã sao lưu thành công lên Drive để lần sau không cần trích xuất lại!")
            except Exception as e:
                print(f"⚠️ Không thể sao lưu lên Drive: {e}")

### Bước 6: Chạy Thực Nghiệm

Bạn có thể chọn chạy từng mô hình riêng biệt hoặc chạy toàn bộ bộ thử nghiệm đối chứng tự động.

#### Lựa chọn A: Chạy Tự Động Toàn Bộ Benchmark Đối Chứng (TTT Baseline vs UC-TTT Proposed)

In [ ]:
!python -u experiments/run_paper_sync_benchmarks.py \
    --embeddings_path data/processed/text_embeddings_bert.pt \
    --window_mode fixed_posts \
    --window_size 8 \
    --fusion_type residual_gate \
    --epochs 35 \
    --patience 8

#### Lựa chọn B: Hoặc Chạy Riêng Lẻ Từng Mô Hình

In [ ]:
# Chạy riêng TTT Baseline (Wu et al. 2025)
!python experiments/train_ttt_baseline.py \
    --embeddings_path data/processed/text_embeddings_bert.pt \
    --events_mode 5 \
    --window_mode fixed_posts \
    --window_size 8 \
    --d_model 768 \
    --nhead 12 \
    --num_tree_layers 6 \
    --epochs 35 \
    --patience 8

In [ ]:
# Chạy riêng UC-TTT Proposed (với Residual-Gate Fusion)
!python experiments/train_ucttt.py \
    --embeddings_path data/processed/text_embeddings_bert.pt \
    --events_mode 5 \
    --window_mode fixed_posts \
    --window_size 8 \
    --fusion_type residual_gate \
    --d_model 768 \
    --nhead 12 \
    --num_tree_layers 6 \
    --epochs 35 \
    --patience 8

### Bước 7: Sao Lưu Kết Quả & Checkpoints Về Google Drive

In [ ]:
# Xác định thư mục sao lưu trên Drive
if DRIVE_DATA_DIR and os.path.exists(DRIVE_DATA_DIR):
    BACKUP_DIR = os.path.join(DRIVE_DATA_DIR, 'results_backup')
else:
    BACKUP_DIR = '/content/drive/MyDrive/PHEME_results_backup'

os.makedirs(BACKUP_DIR, exist_ok=True)

!cp -r results/* "$BACKUP_DIR/"
print(f'✅ Đã sao lưu toàn bộ kết quả và bảng so sánh vào: {BACKUP_DIR}')